<a href="https://colab.research.google.com/github/milicaclm/transportni_problem/blob/main/transportni_problem_projekat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
import numpy as np
import pandas as pd

In [37]:
kapacitet = [20,30, 20, 10]
potražnja = [10, 40, 30]

cene = np.array([
    [10, 12, 0],
    [8, 4, 3],
    [6, 9, 4],
    [7, 8, 5]
])

cost_df = pd.DataFrame(cene, columns=['D1', 'D2', 'D3'], index=['F1', 'F2', 'F3', 'F4'])

print("Kapaciteti (Supply):", kapacitet)
print("Zahtevi (Demand):", potražnja)
print("Troškovi transporta (Costs):")
print(cost_df)

Kapaciteti (Supply): [20, 30, 20, 10]
Zahtevi (Demand): [10, 40, 30]
Troškovi transporta (Costs):
    D1  D2  D3
F1  10  12   0
F2   8   4   3
F3   6   9   4
F4   7   8   5


In [38]:
def severozapadni_ugao(kapacitet, potražnja):

    resenje = np.zeros((len(kapacitet), len(potražnja)))

    kap = kapacitet.copy()
    pot = potražnja.copy()

    i = 0
    j = 0

    while i < len(kapacitet) and j < len(potražnja):
        if kap[i] <= pot[j]:
            resenje[i][j] = kap[i]
            pot[j] -= kap[i]
            i += 1
        else:
            resenje[i][j] = pot[j]
            kap[i] -= pot[j]
            j += 1

    return resenje

In [39]:
def metod_najmanjih_cena(kapacitet, potražnja, cene):

    resenje = np.zeros((len(kapacitet), len(potražnja)))

    kap = kapacitet.copy()
    pot = potražnja.copy()

    cost_indices = [(cene[i][j], i, j) for i in range(len(kapacitet)) for j in range(len(potražnja))]
    cost_indices.sort()

    for cost, i, j in cost_indices:
        if kap[i] == 0 or pot[j] == 0:
            continue
        allocation = min(kap[i], pot[j])
        resenje[i][j] = allocation
        kap[i] -= allocation
        pot[j] -= allocation

    return resenje

In [40]:
def fogelov_metod(kapacitet, potražnja, costs):

    resenje = np.zeros((len(kapacitet), len(potražnja)))

    kap = kapacitet.copy()
    pot = potražnja.copy()
    cos = costs.copy()

    INF = int(1e10)

    while np.sum(kap) > 0 and np.sum(pot) > 0:

        redovni_penali = []
        for i in range(len(kap)):
            if kap[i] == 0:
                redovni_penali.append(-INF)
            else:
                red = sorted([costs[i][j] for j in range(len(pot)) if pot[j] > 0])
                if len(red) > 1:
                    redovni_penali.append(red[1] - red[0])
                else:
                    redovni_penali.append(red[0] if len(red) > 0 else -INF)

        kolonski_penali = []
        for j in range(len(pot)):
            if pot[j] == 0:
                kolonski_penali.append(-INF)
            else:
                kolona = sorted([cos[i][j] for i in range(len(kap)) if kap[i] > 0])
                if len(kolona) > 1:
                    kolonski_penali.append(kolona[1] - kolona[0])
                else:
                    kolonski_penali.append(kolona[0] if len(kolona) > 0 else -INF)

        max_red_penal = max(redovni_penali)
        max_kol_penal = max(kolonski_penali)

        if max_red_penal > max_kol_penal:
            i = redovni_penali.index(max_red_penal)
            j = np.argmin([cos[i][j] if pot[j] > 0 else INF for j in range(len(pot))])
        else:
            j = kolonski_penali.index(max_kol_penal)
            i = np.argmin([cos[i][j] if kap[i] > 0 else INF for i in range(len(kap))])

        allocation = min(kap[i], pot[j])
        resenje[i][j] = allocation
        kap[i] -= allocation
        pot[j] -= allocation

        if kap[i] == 0:
            for k in range(len(pot)):
                cos[i][k] = INF
        if pot[j] == 0:
            for k in range(len(kap)):
                cos[k][j] = INF

    return resenje

In [41]:
resenje_severozapadni = severozapadni_ugao(kapacitet, potražnja)
resenje_severozapadni_df = pd.DataFrame(resenje_severozapadni, columns=cost_df.columns, index=cost_df.index)

print("Inicijalno rešenje (Severozapadni ugao):")
print(resenje_severozapadni_df)

resenje_najmanje_cene = metod_najmanjih_cena(kapacitet, potražnja, cene)
resenje_najmanje_cene_df = pd.DataFrame(resenje_najmanje_cene, columns=cost_df.columns, index=cost_df.index)

print("Inicijalno rešenje (Metod najmanjih cena):")
print(resenje_najmanje_cene_df)

resenje_fogelov = fogelov_metod(kapacitet, potražnja, cene)
resenje_fogelov_df = pd.DataFrame(resenje_fogelov, columns=cost_df.columns, index=cost_df.index)

print("Inicijalno rešenje (Fogelov metod):")
print(resenje_fogelov_df)

Inicijalno rešenje (Severozapadni ugao):
      D1    D2    D3
F1  10.0  10.0   0.0
F2   0.0  30.0   0.0
F3   0.0   0.0  20.0
F4   0.0   0.0  10.0
Inicijalno rešenje (Metod najmanjih cena):
      D1    D2    D3
F1   0.0   0.0  20.0
F2   0.0  20.0  10.0
F3  10.0  10.0   0.0
F4   0.0  10.0   0.0
Inicijalno rešenje (Fogelov metod):
      D1    D2    D3
F1   0.0   0.0  20.0
F2   0.0  30.0   0.0
F3  10.0   0.0  10.0
F4   0.0  10.0   0.0


In [ ]:
def iterativni_postupak(cene: np.ndarray, inicijalno_resenje: pd.DataFrame, ispisi_postupak: bool = False):
    u = [None for idx in inicijalno_resenje.index]
    v = [None for col in inicijalno_resenje.columns]
    degenerativne_nule = []
    i_min = None
    j_min = None
    c_prim = None
    resenje = inicijalno_resenje.where(inicijalno_resenje != 0, other=np.nan)
    indexi = inicijalno_resenje.index
    kolone = inicijalno_resenje.columns
    resenje = pd.DataFrame(resenje.values)
    pozitivne_indeksi = []
    najpopunjeniji_red = None  # Inicijalizacija sa prvim redom
    # Stavlja nulu u 'u' za najpopunjeniji red inicijalnog resenja (sto manje praznih polja)

    # c[i][j] == u[i] + v[j]
    def korak1():
        nonlocal u, v, najpopunjeniji_red, resenje
        u = [None] * inicijalno_resenje.shape[0]
        v = [None] * inicijalno_resenje.shape[1]
        nove_degenerativne = []
        
        #dodavanje degenerativnih nula tako da ima m+n-1 not np.nan polja u resenju
        najpopunjeniji_red = (resenje.notna().sum(axis=1)).idxmax()
        while resenje.notna().sum().sum() < inicijalno_resenje.shape[0] + inicijalno_resenje.shape[1] - 1:
            if ispisi_postupak:
                print ("broj popunjenih", (resenje.notna()).sum().sum(), "m+n-1:", inicijalno_resenje.shape[0] + inicijalno_resenje.shape[1] - 1)
                print("resenje:")
                print(resenje)
            dodata = False
            for j in resenje.columns:
                if pd.isna(resenje.iloc[najpopunjeniji_red, j]):
                    resenje.iloc[najpopunjeniji_red, j] = 0
                    nove_degenerativne.append((najpopunjeniji_red, j))
                    dodata = True
                    break
            if not dodata:
                
                for i in resenje.index:
                    for j in resenje.columns:
                        if pd.isna(resenje.iloc[i, j]):
                            resenje.iloc[i, j] = 0
                            nove_degenerativne.append((i, j))
                            dodata = True
                            break
            if not dodata:
                raise ValueError("Nije moguće dodati degenerativnu nulu, proverite inicijalno rešenje.")
            
        najpopunjeniji_red = (resenje.notna().sum(axis=1)).idxmax()
        najpopunjeniji_red = resenje.index.get_loc(najpopunjeniji_red)
        u[najpopunjeniji_red] = 0



        for _ in range(len(resenje.index) + len(resenje.columns)):
            for i in resenje.index:
                for j in resenje.columns:
                    
                    if pd.notna(resenje.iloc[i, j]) and (i, j) not in degenerativne_nule:
                        if (u[i] is None) and (v[j] is not None):
                            u[i] = cene[i, j] - v[j]
                        elif (v[j] is None) and (u[i] is not None):
                            v[j] = cene[i, j] - u[i]
            if all(val is not None for val in u) and all(val is not None for val in v):
                degenerativne_nule.extend(nove_degenerativne)
                if ispisi_postupak:
                    print("KORAK 1:")
                    print("najpopunjeniji_red:", najpopunjeniji_red)
                    print("u=", u)
                    print("v=", v)
                    print("resenje:")
                    print(resenje)
                return
        if any(val is None for val in u) or any(val is None for val in v):
            print("v = ", v)
            print("u = ", u)
            print("najpopunjeniji red:", najpopunjeniji_red)
            print("Inicijalno rešenje:")
            print(resenje)
            raise ValueError("Nedostaju vrednosti u[i] ili v[j]")

    def korak2():
        nonlocal j_min, i_min, pozitivne_indeksi, c_prim
        pozitivne_indeksi.clear()
        j_min = None
        i_min = None
        c_prim = np.zeros_like(cene)
        for i in resenje.index: 
            for j in resenje.columns:
                if pd.isna(resenje.iloc[i, j]):
                    if (u[i] is not None) and (v[j] is not None):

                        c_prim[i, j] = cene[i, j] - u[i] - v[j]
                        if c_prim[i, j] < 0 and (i_min is None or j_min is None or c_prim[i, j] < c_prim[i_min, j_min]):
                            #print(f"Ažuriram i_min={i}, j_min={j}, c_prim[i, j]={c_prim[i, j]}")
                            i_min = i
                            j_min = j
                    else:
                        raise ValueError("Nedostaju vrednosti u[i] ili v[j] za indeks {} i kolonu {}".format(i, j))
                else:
                    pozitivne_indeksi.append((i, j))
        if ispisi_postupak:
            print("KORAK 2:")
            print("c_prim:")
            print(c_prim)
            print("i_min:", i_min, "j_min:", j_min)
            print("pozitivne_indeksi:", pozitivne_indeksi)
            print("cene:")
            print(cene)
        return (i_min is not None) and (j_min is not None)
    
    def korak3():
        nonlocal resenje, pozitivne_indeksi
        pozitivne_indeksi += [(i_min, j_min)] 
        
        """def rekurzivna_pretraga(i, j, put, is_row): #trazi put od i_min, j_min do i_min, j_min
     
            if i == i_min and j == j_min and len(put) > 3 and len(put) % 2 == 0:
                return put
            for i_sl, j_sl in pozitivne_indeksi:
                if (i_sl, j_sl) in put:
                    continue
                if len(put) % 2 != 0 and resenje.iloc[i_sl, j_sl] == 0:
                    continue
                novi_put = put + [(i_sl, j_sl)]

                if is_row and i_sl == i and j_sl != j:
                    rez = rekurzivna_pretraga(i_sl, j_sl, novi_put, False)
                    if rez is not None:
                        return rez
                elif not is_row and j_sl == j and i_sl != i:
                    rez = rekurzivna_pretraga(i_sl, j_sl, novi_put, True)
                    if rez is not None:
                        return rez
            
            return None"""
        

        def rekurzivna_pretraga(put, is_row):
            if len(put) > len(pozitivne_indeksi) + 1:
                return None 
            if (put[-1][0], put[-1][1]) == (i_min, j_min) and len(put) > 4 and len(put) % 2 != 0:
                return put[:-1]
                
            for i_sl, j_sl in pozitivne_indeksi:
                # Ne moze biti naredni cvor ako se do njega dolazi dijagonalno
                if i_sl != put[-1][0] and j_sl != put[-1][1]:
                    continue
                # Degenerativna nula ne moze biti na minus poziciji
                if len(put) % 2 != 0 and (i_sl, j_sl) in degenerativne_nule:
                    continue
                # Ne moze da bude već u putu osim ako nije početni čvor
                if (i_sl, j_sl) in put and (i_sl, j_sl) != (i_min, j_min):
                    continue
                # Nije vreme da se vrati u pocetni cvor kada nije potpuna kontura
                if (i_sl, j_sl) == (i_min, j_min) and len(put) < 4:
                    continue
                
                # Mora biti teme konture da bi se dodao u put
                if not is_row and i_sl == put[-1][0] and j_sl != put[-1][1]:
                    continue
                if is_row and j_sl == put[-1][1] and i_sl != put[-1][0]: 
                    continue
                    
                novi_put = put + [(i_sl, j_sl)]
                    
                rez = rekurzivna_pretraga(novi_put, not is_row)
                if rez is not None:

                    return rez
                
            return None

                    
        def novo_resenje(put):
            nonlocal resenje
            min_oznaka = min(resenje.iloc[i, j] for i, j in put if put.index((i,j)) % 2 != 0)
            resenje.iloc[i_min, j_min] = min_oznaka
            for i, j in put:
                if put.index((i,j)) % 2 != 0:
                    resenje.iloc[i, j] += -min_oznaka
                    if resenje.iloc[i, j] == 0:
                        resenje.iloc[i, j] = np.nan
                    if resenje.iloc[i, j] < 0:
                        raise ValueError(f"Negativna vrednost u resenju na indeksu ({i}, {j}): {resenje.iloc[i, j]}")
                else:
                    resenje.iloc[i, j] += min_oznaka
                    if (i,j) in degenerativne_nule and min_oznaka > 0:
                        degenerativne_nule.remove((i, j))
            

        
        put = rekurzivna_pretraga([(i_min, j_min)], True) or rekurzivna_pretraga([(i_min, j_min)], False)
        if ispisi_postupak:
            print("KORAK 3:")        
            print("Pronađen put:", put)
            print("Novo rešenje:")
            print(resenje)
        
        if put is None:
            print("")
            raise ValueError("Nije pronađen validan put za optimizaciju.")
        else:
            novo_resenje(put)
        
    count = 0
    while True:
        if ispisi_postupak: 
            print(f"ITERACIJA {count + 1}")
        korak1()
        if not korak2():
            resenje = resenje.fillna(0)
            resenje.index = indexi
            resenje.columns = kolone
            return resenje
        korak3()
        count += 1
        if count > 10:
            raise ValueError("Previše iteracija, proverite inicijalno rešenje.")
        


tabela_resenje_severozapadni = iterativni_postupak(cene, resenje_severozapadni_df, True)
print("Optimalno rešenje (Severozapadni ugao):")
print(tabela_resenje_severozapadni)
tabela_resenje_najmanje_cene = iterativni_postupak(cene, resenje_najmanje_cene_df)
print("Optimalno rešenje (Metod najmanjih cena):")
print(tabela_resenje_najmanje_cene)
tabela_resenje_fogelov = iterativni_postupak(cene, resenje_fogelov_df)
print("Optimalno rešenje (Fogelov metod):")
print(tabela_resenje_fogelov)


ITERACIJA 1
broj popunjenih 5 m+n-1: 6
resenje:
      0     1     2
0  10.0  10.0   NaN
1   NaN  30.0   NaN
2   NaN   NaN  20.0
3   NaN   NaN  10.0
KORAK 1:
najpopunjeniji_red: 0
u= [0, np.int64(-8), np.int64(4), np.int64(5)]
v= [np.int64(10), np.int64(12), np.int64(0)]
resenje:
      0     1     2
0  10.0  10.0   0.0
1   NaN  30.0   NaN
2   NaN   NaN  20.0
3   NaN   NaN  10.0
KORAK 2:
c_prim:
[[ 0  0  0]
 [ 6  0 11]
 [-8 -7  0]
 [-8 -9  0]]
i_min: 3 j_min: 1
pozitivne_indeksi: [(0, 0), (0, 1), (0, 2), (1, 1), (2, 2), (3, 2)]
cene:
[[10 12  0]
 [ 8  4  3]
 [ 6  9  4]
 [ 7  8  5]]
KORAK 3:
Pronađen put: [(3, 1), (3, 2), (0, 2), (0, 1)]
Novo rešenje:
      0     1     2
0  10.0  10.0   0.0
1   NaN  30.0   NaN
2   NaN   NaN  20.0
3   NaN   NaN  10.0
ITERACIJA 2
broj popunjenih 5 m+n-1: 6
resenje:
      0     1     2
0  10.0   NaN  10.0
1   NaN  30.0   NaN
2   NaN   NaN  20.0
3   NaN  20.0   NaN
KORAK 1:
najpopunjeniji_red: 0
u= [0, np.int64(-8), np.int64(4), np.int64(-4)]
v= [np.int64(10)

ValueError: Nedostaju vrednosti u[i] ili v[j]